# Um token opcional para os rates de download

In [1]:
from dotenv import load_dotenv
import os

load_dotenv() # .env
token = os.getenv('HF_TOKEN')

if token is None:
    print("HF_TOKEN não encontrado no .env — a continuar sem token")
else:
    os.environ['HF_TOKEN'] = token
    print("HF_TOKEN carregado")


HF_TOKEN carregado


# Imports

In [2]:
import pandas as pd
import numpy as np
import re
from datasets import load_dataset,disable_caching
import uuid

In [3]:
# para quand o oespaço não ajuda
disable_caching()

# Load e Processamento dos dados

## 1. Carregar e processar `gsingh1-py/train e teste`

In [4]:
df1_train = load_dataset('gsingh1-py/train', split='train').to_pandas()
df1_test  = load_dataset('gsingh1-py/test',  split='test').to_pandas()
df1 = pd.concat([df1_train, df1_test], ignore_index=True)
# total: 7320 + 1569 = 8889 rows antes do melt
print(f"gsingh1 train: {len(df1_train)}")
print(f"gsingh1 test:  {len(df1_test)}")
print(f"gsingh1 total: {len(df1)}")

Repo card metadata block was not found. Setting CardData to empty.


Generating train split:   0%|          | 0/7321 [00:00<?, ? examples/s]

Repo card metadata block was not found. Setting CardData to empty.


Generating test split:   0%|          | 0/1569 [00:00<?, ? examples/s]

gsingh1 train: 7321
gsingh1 test:  1569
gsingh1 total: 8890


In [5]:
df1.columns

Index(['prompt', 'Human_story', 'gemma-2-9b', 'mistral-7B', 'qwen-2-72B',
       'llama-8B', 'accounts/yi-01-ai/models/yi-large', 'GPT_4-o'],
      dtype='str')

In [6]:
GSINGH_LABEL_MAP = {
    'Human_story': 'human',
    'gemma-2-9b':  'google',
    'mistral-7B':  'mistral',
    'llama-8B':    'meta',
    'GPT_4-o':     'openai'
}

df1_melted = df1.melt(
    value_vars=list(GSINGH_LABEL_MAP.keys()),
    value_name='text'
).rename(columns={'variable': 'label'})

df1_melted['label'] = df1_melted['label'].map(GSINGH_LABEL_MAP)
df1_melted = df1_melted[['text', 'label']].dropna()

print(f"gsingh1 após melt: {len(df1_melted)}")
print(df1_melted['label'].value_counts())


gsingh1 após melt: 44376
label
openai     8890
mistral    8884
google     8878
meta       8871
human      8853
Name: count, dtype: int64


## 2. Carregar e processar `OpenTuringBench`

In [7]:
ds2 = load_dataset('MLNTeam-Unical/OpenTuringBench', 'in_domain')
print(ds2)  # ver splits disponíveis primeiro

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['url', 'content', 'model'],
        num_rows: 231980
    })
    val: Dataset({
        features: ['url', 'content', 'model'],
        num_rows: 29001
    })
    test: Dataset({
        features: ['url', 'content', 'model'],
        num_rows: 29001
    })
})


In [8]:
ds2 = load_dataset('MLNTeam-Unical/OpenTuringBench', 'in_domain')
df2 = pd.concat([
    ds2['train'].to_pandas(),
    ds2['val'].to_pandas(),
    ds2['test'].to_pandas()
], ignore_index=True)

df2 = df2.drop(columns=['url']) 
print(df2.columns)

Index(['content', 'model'], dtype='str')


In [9]:
print(df2.head())
print(df2['model'].unique())

                                             content  \
0  3 Signs You Should Call Off The Wedding\n\nDec...   
1  Robert De Niro Goes Full ‘Taxi Driver’ On Peop...   
2  New York Times Journalists Who Broke Harvey We...   
3    "A Beacon of Hope: Transgender Woman Wins Le...   
4   There Is No Debate: The Tyranny of the Mobili...   

                                model  
0    meta-llama/Llama-3.1-8B-Instruct  
1            Qwen/Qwen2.5-7B-Instruct  
2            Qwen/Qwen2.5-7B-Instruct  
3  mistralai/Mistral-7B-Instruct-v0.3  
4                google/gemma-2-9b-it  
<ArrowStringArray>
[  'meta-llama/Llama-3.1-8B-Instruct',           'Qwen/Qwen2.5-7B-Instruct',
 'mistralai/Mistral-7B-Instruct-v0.3',               'google/gemma-2-9b-it',
          'Intel/neural-chat-7b-v3-3',    'microsoft/Phi-3.5-mini-instruct',
  'upstage/SOLAR-10.7B-Instruct-v1.0']
Length: 7, dtype: str


In [10]:
OTB_LABEL_MAP = {
    'meta-llama/Llama-3.1-8B-Instruct':   'meta',
    'mistralai/Mistral-7B-Instruct-v0.3': 'mistral',
    'google/gemma-2-9b-it':               'google',
}

df2['label'] = df2['model'].map(OTB_LABEL_MAP)
df2 = df2[df2['label'].notna()].reset_index(drop=True)
df2 = df2.rename(columns={'content': 'text'})[['text', 'label']]

print(f"OpenTuringBench após filtro: {len(df2)}")
print(df2['label'].value_counts())

OpenTuringBench após filtro: 124278
label
meta       41426
mistral    41426
google     41426
Name: count, dtype: int64


## 3. Carregar e Processar o `RAID`

In [11]:
ds3 = load_dataset('liamdugan/raid', split='train')
df3 = ds3.select_columns(['id', 'model', 'generation']).to_pandas()
print(f"RAID total: {len(df3)}")
print(df3.columns)

train.csv:   0%|          | 0.00/11.8G [00:00<?, ?B/s]

c:\Users\bruno\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bruno\.cache\huggingface\hub\datasets--liamdugan--raid. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


extra.csv:   0%|          | 0.00/3.71G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating extra split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

RAID total: 5615820
Index(['id', 'model', 'generation'], dtype='str')


In [12]:
df3.head()

,id,model,generation
0,e5e058ce-be2b-459d-af36-32532aaba5ff,human,The recent advancements in artificial intellig...
1,f95b107b-d176-4af5-90f7-4d0bb20caf93,human,High-quality training data play a key role in ...
2,856d8972-9e3d-4544-babc-0fe16f21e04d,human,The success of deep learning methods in medica...
3,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,human,Simultaneous segmentation of multiple organs f...
4,72c41b8d-0069-4886-b734-a4000ffca286,human,Detection faults in seismic data is a crucial ...


In [13]:
df3["model"].unique()

<ArrowStringArray>
[       'human',   'llama-chat',          'mpt',     'mpt-chat',
         'gpt2',      'mistral', 'mistral-chat',         'gpt3',
       'cohere',      'chatgpt',         'gpt4',  'cohere-chat']
Length: 12, dtype: str

In [14]:
RAID_LABEL_MAP = {
    'human':         'human',
    'mistral':       'mistral',
    'mistral-chat':  'mistral',
    'gpt3':          'openai',
    'chatgpt':       'openai',
    'gpt4':          'openai',
    'llama-chat':    'meta',
}

df3['label'] = df3['model'].map(RAID_LABEL_MAP)
df3 = df3[df3['label'].notna()].reset_index(drop=True)
df3 = df3.rename(columns={'generation': 'text'})[['text', 'label']]

print(f"RAID após filtro: {len(df3)}")
print(df3['label'].value_counts())

RAID após filtro: 3048588
label
mistral    1283616
openai      962712
meta        641808
human       160452
Name: count, dtype: int64


# Merge de todos os dados

In [15]:
print(df1_melted.head())
print(df2.head())
print(df3.head())

                                                text  label
0  Comments\nThe U.S. bombings thatended World Wa...  human
1  new video loaded:Messages From Quarantine\ntra...  human
2  Supported by\nRoberta Karmel, First Woman Name...  human
3  Supported by\nContests\nSummer Reading Contest...  human
4  The Week on Instagram\n@heislerphoto was one o...  human
                                                text    label
0  3 Signs You Should Call Off The Wedding\n\nDec...     meta
1    "A Beacon of Hope: Transgender Woman Wins Le...  mistral
2   There Is No Debate: The Tyranny of the Mobili...   google
3   Kids Weigh In On The 2012 Election (VIDEO)\n\...   google
4    Man Sentenced to Five Years in Prison for Ta...  mistral
                                                text  label
0  The recent advancements in artificial intellig...  human
1  High-quality training data play a key role in ...  human
2  The success of deep learning methods in medica...  human
3  Simultaneous segmentation

In [16]:
print("##################\ngsingh1\n##################")
print(df1_melted['label'].value_counts())
print("##################\nOpenTuringBench\n##################")
print(df2['label'].value_counts())
print("##################\nRaid\n##################")
print(df3['label'].value_counts())

##################
gsingh1
##################
label
openai     8890
mistral    8884
google     8878
meta       8871
human      8853
Name: count, dtype: int64
##################
OpenTuringBench
##################
label
meta       41426
mistral    41426
google     41426
Name: count, dtype: int64
##################
Raid
##################
label
mistral    1283616
openai      962712
meta        641808
human       160452
Name: count, dtype: int64


In [17]:
df_all = pd.concat([df1_melted, df2, df3], ignore_index=True)
print(f"Total antes de sampling: {len(df_all)}")
print(df_all['label'].value_counts())

Total antes de sampling: 3217242
label
mistral    1333926
openai      971602
meta        692105
human       169305
google       50304
Name: count, dtype: int64


In [18]:
df_all.head()

,text,label
0,Comments\nThe U.S. bombings thatended World Wa...,human
1,new video loaded:Messages From Quarantine\ntra...,human
2,"Supported by\nRoberta Karmel, First Woman Name...",human
3,Supported by\nContests\nSummer Reading Contest...,human
4,The Week on Instagram\n@heislerphoto was one o...,human


### Caso se queira guardar todos os dados em um dataset para uso

In [ ]:
# OPCIONAL: Guardar os datasets

# df1_melted.to_parquet('gsingh1.parquet', index=False)
# df2.to_parquet('open_turing.parquet', index=False)  
# df3.to_parquet('raid.parquet', index=False)
# df_all.to_parquet('all.parquet', index=False)

# print("datasets guardados")

## Criar um dataset balanceado entre as classes, 40k linhas para cada classe

In [19]:
MAX_PER_CLASS = 40_000

dfs = []
for label in df_all['label'].unique():
    df_label = df_all[df_all['label'] == label]
    dfs.append(df_label.sample(n=min(len(df_label), MAX_PER_CLASS), random_state=808815))

df_final = pd.concat(dfs, ignore_index=True)
df_final = df_final.sample(frac=1, random_state=808815).reset_index(drop=True)

print(f"Total após sampling: {len(df_final)}")
print(df_final['label'].value_counts())

Total após sampling: 200000
label
openai     40000
mistral    40000
google     40000
meta       40000
human      40000
Name: count, dtype: int64


# Guardar os dados em formato parquet

In [20]:
df_final.to_parquet('dataset_40k.parquet', index=False)
print(df_final.info())

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    200000 non-null  str  
 1   label   200000 non-null  str  
dtypes: str(2)
memory usage: 390.9 MB
None


# Eliminar os datasets da cache

In [21]:
# Continuar no problema de espaço para quem quiser, elimina a chace, que ocumula bastante ainda
import shutil
import os
datasets_to_clear = [
    'gsingh1-py___train',
    'gsingh1-py___test', 
    'MLNTeam-Unical___open_turing_bench',
    'liamdugan___raid'
]

cache_dir = os.path.expanduser('~/.cache/huggingface/datasets')
for ds in datasets_to_clear:
    path = os.path.join(cache_dir, ds)
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Removido: {ds}")
    else:
        print(f"Não encontrado: {ds}")

Removido: gsingh1-py___train
Removido: gsingh1-py___test
Removido: MLNTeam-Unical___open_turing_bench
Removido: liamdugan___raid


# Load do Dataset tratado depois do upload

In [22]:
from datasets import load_dataset

df = load_dataset("BrunoFilipeRDS/ap-40k-balanced-blend", split='train').to_pandas()
print(f"Dataset carregado: {len(df)} rows")
print(df['label'].value_counts())

README.md:   0%|          | 0.00/328 [00:00<?, ?B/s]

c:\Users\bruno\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bruno\.cache\huggingface\hub\datasets--BrunoFilipeRDS--ap-40k-balanced-blend. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Dataset carregado: 200000 rows
label
openai     40000
mistral    40000
google     40000
meta       40000
human      40000
Name: count, dtype: int64


In [24]:
import shutil
import os
datasets_to_clear = [
    'BrunoFilipeRDS___ap-40k-balanced-blend'
]

cache_dir = os.path.expanduser('~/.cache/huggingface/datasets')
for ds in datasets_to_clear:
    path = os.path.join(cache_dir, ds)
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Removido: {ds}")
    else:
        print(f"Não encontrado: {ds}")

Removido: BrunoFilipeRDS___ap-40k-balanced-blend
